# Credit Card Fraud Detection with Benford's Law

Multi-model comparison on the ULB credit card fraud dataset. Inspired by Stripe's Payments Foundation Model (sequence-aware fraud detection) and classical forensic accounting (Benford's Law).

**Run All** on Colab to reproduce.

## 0. Bootstrap

In [ ]:
import os, sys

REPO_URL = "https://github.com/dutch-casa/fraud-benford.git"

if "google.colab" in sys.modules:
    if not os.path.exists("fraud-benford"):
        !git clone {REPO_URL}
    %cd fraud-benford
    !pip install -q -r requirements.txt

repo_root = os.path.abspath("." if os.path.basename(os.getcwd()) == "fraud-benford" else "..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.data import load_raw, time_ordered_split
from src.benford import (
    BENFORD_PROBS,
    benford_features,
    chi_square_distance,
    empirical_distribution,
    leading_digit_series,
)
from src.features import build_feature_matrix
from src.evaluation import (
    evaluate,
    model_comparison_table,
    plot_benford_histogram,
    plot_pr_curves,
    plot_roc_curves,
)
from src.models import (
    autoencoder_anomaly,
    lightgbm_classifier,
    logistic_regression,
    mlp_classifier,
    xgboost_classifier,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = [10, 6]
RANDOM_SEED = 42

## 1. Problem Definition and Scope

Detect fraudulent credit card transactions on the ULB public dataset (284,807 transactions over ~48 hours, 492 labeled as fraud — a 0.17% positive rate).

**Objectives (SMART):**
- **Specific:** binary classification of fraud vs. legitimate transactions.
- **Measurable:** area under the precision-recall curve (AUPRC), precision at recall ≥ 0.80.
- **Achievable:** published baselines on this dataset report AUPRC ≈ 0.75 for strong models.
- **Relevant:** mirrors the real-world task Stripe Radar and similar systems solve.
- **Time-bound:** one Colab Run-All session.

**Approach:** staircase progression from a weak linear baseline to gradient-boosted trees, a small MLP, and a torch autoencoder, with Benford's Law features bolted on at one step to measure their marginal contribution.

## 2. Data Loading and Exploration

In [ ]:
df = load_raw()
print(f"shape: {df.shape}")
print(f"fraud rate: {df['Class'].mean():.4%}")
print(f"time range: {df['Time'].min():.0f}s \u2192 {df['Time'].max():.0f}s ({df['Time'].max() / 3600:.1f} hours)")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

counts = df["Class"].value_counts()
axes[0].bar(["legit", "fraud"], counts.values, color=["#4c72b0", "#c44e52"])
axes[0].set_yscale("log")
axes[0].set_title("Class distribution (log scale)")
axes[0].set_ylabel("Count")

axes[1].hist(df.loc[df["Class"] == 0, "Amount"], bins=60, alpha=0.6, label="legit", log=True)
axes[1].hist(df.loc[df["Class"] == 1, "Amount"], bins=60, alpha=0.8, label="fraud", log=True, color="#c44e52")
axes[1].set_title("Amount distribution")
axes[1].set_xlabel("Amount")
axes[1].set_ylabel("Count (log)")
axes[1].legend()

axes[2].hist(df.loc[df["Class"] == 0, "Time"] / 3600, bins=48, alpha=0.6, label="legit")
axes[2].hist(df.loc[df["Class"] == 1, "Time"] / 3600, bins=48, alpha=0.8, label="fraud", color="#c44e52")
axes[2].set_title("Transactions over time")
axes[2].set_xlabel("Hours since first transaction")
axes[2].set_ylabel("Count")
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Preprocessing and Feature Engineering

### 3a. Benford's Law

Benford's Law says the leading digit $d$ of naturally-occurring numerical data follows $P(d) = \log_{10}(1 + 1/d)$ — digit 1 appears ~30% of the time, digit 9 only ~4.6%. Real transaction amounts tend to obey it. Fabricated or anomalous amounts often don't. We use this as both a diagnostic plot and a feature.

In [ ]:
digits_all = leading_digit_series(df["Amount"])
digits_legit = leading_digit_series(df.loc[df["Class"] == 0, "Amount"])
digits_fraud = leading_digit_series(df.loc[df["Class"] == 1, "Amount"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_benford_histogram(digits_all, "All transactions", ax=axes[0])
plot_benford_histogram(digits_legit, "Legitimate", ax=axes[1])
plot_benford_histogram(digits_fraud, "Fraud", ax=axes[2])
plt.tight_layout()
plt.show()

for label, digits in [("all", digits_all), ("legit", digits_legit), ("fraud", digits_fraud)]:
    chi2 = chi_square_distance(empirical_distribution(digits))
    print(f"chi-square distance from Benford ({label}): {chi2:.4f}")

### 3b. Time-window aggregates

Inspired by Stripe's observation that card-testing attacks are bursty and sequential. We build rolling-window features over the `Time` axis: transaction counts in 60s/600s windows, rolling amount sums, time-since-last, and a rolling amount z-score.

In [ ]:
time_features = build_feature_matrix(df)
print(f"time-window feature shape: {time_features.shape}")
time_features.head()

In [ ]:
benford_feats = benford_features(df)
print(f"benford feature shape: {benford_feats.shape}")
benford_feats.head()

## 4. Model Development

We split the data **time-ordered** (no random shuffle) to avoid future-leakage — this is how real fraud systems are evaluated. The test set is the last 20% of transactions in chronological order.

### Feature sets

- **base** — the 28 PCA features + Amount (what every naive classifier gets)
- **+benford** — base plus 9 one-hot leading-digit features
- **+time** — base plus Benford plus 5 time-window aggregate features ("full")

In [ ]:
base_cols = [f"V{i}" for i in range(1, 29)] + ["Amount"]

features_base = df[base_cols].copy()
features_benford = pd.concat([features_base, benford_feats], axis=1)
features_full = pd.concat([features_benford, time_features], axis=1)
y = df["Class"].astype(int)

split_idx = int(len(df) * 0.8)
print(f"train rows: {split_idx}, test rows: {len(df) - split_idx}")
print(f"train fraud: {y.iloc[:split_idx].sum()}, test fraud: {y.iloc[split_idx:].sum()}")

In [ ]:
def split_xy(X: pd.DataFrame, y: pd.Series, cut: int) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    return X.iloc[:cut], y.iloc[:cut], X.iloc[cut:], y.iloc[cut:]

def train_and_score(name: str, model, X: pd.DataFrame, y: pd.Series, cut: int) -> tuple[np.ndarray, object]:
    X_tr, y_tr, X_te, y_te = split_xy(X, y, cut)
    model.fit(X_tr, y_tr)
    scores = model.predict_proba(X_te)[:, 1]
    report = evaluate(y_te, scores)
    print(f"  {name:40s} AUPRC={report.auprc:.4f}  ROC-AUC={report.roc_auc:.4f}  P@R=0.80={report.precision_at_recall_80:.4f}")
    return scores, report

In [ ]:
reports: dict = {}
scored: dict = {}
y_test = y.iloc[split_idx:]

print("training models...")

name = "1. LR (base)"
scores, reports[name] = train_and_score(name, logistic_regression(), features_base, y, split_idx)
scored[name] = (y_test, scores)

name = "2. LR (base, balanced)"
scores, reports[name] = train_and_score(name, logistic_regression(class_weight="balanced"), features_base, y, split_idx)
scored[name] = (y_test, scores)

name = "3. LR (+ Benford, balanced)"
scores, reports[name] = train_and_score(name, logistic_regression(class_weight="balanced"), features_benford, y, split_idx)
scored[name] = (y_test, scores)

name = "4. LR (full, balanced)"
scores, reports[name] = train_and_score(name, logistic_regression(class_weight="balanced"), features_full, y, split_idx)
scored[name] = (y_test, scores)

n_train_frauds = int(y.iloc[:split_idx].sum())
n_train_legit = split_idx - n_train_frauds
spw = n_train_legit / max(n_train_frauds, 1)

name = "5. XGBoost (full)"
scores, reports[name] = train_and_score(name, xgboost_classifier(scale_pos_weight=spw), features_full, y, split_idx)
scored[name] = (y_test, scores)

name = "6. LightGBM (full)"
scores, reports[name] = train_and_score(name, lightgbm_classifier(), features_full, y, split_idx)
scored[name] = (y_test, scores)

name = "7. MLP (full, scaled)"
scores, reports[name] = train_and_score(name, make_pipeline(StandardScaler(), mlp_classifier()), features_full, y, split_idx)
scored[name] = (y_test, scores)

name = "8. Autoencoder anomaly (full)"
scores, reports[name] = train_and_score(name, autoencoder_anomaly(), features_full, y, split_idx)
scored[name] = (y_test, scores)

## 5. Evaluation

In [ ]:
comparison = model_comparison_table(reports)
comparison.style.format({"AUPRC": "{:.4f}", "ROC-AUC": "{:.4f}", "P@R=0.80": "{:.4f}", "Best F1": "{:.4f}", "Threshold": "{:.4f}"})

In [ ]:
plot_pr_curves(scored)

In [ ]:
plot_roc_curves(scored)

In [ ]:
best_name = comparison.iloc[0]["model"]
best = reports[best_name]
print(f"best model by AUPRC: {best_name}")
print(f"  AUPRC: {best.auprc:.4f}")
print(f"  ROC-AUC: {best.roc_auc:.4f}")
print(f"  best F1: {best.best_f1:.4f} at threshold {best.best_f1_threshold:.4f}")
print(f"  confusion matrix (rows=true, cols=pred, order=[legit, fraud]):")
print(best.confusion_at_best_f1)

## 6. Discussion and Limitations

### What we learned
- **Class weighting matters more than any single feature.** Moving from the unbalanced LR baseline to `class_weight="balanced"` is typically the single biggest jump in AUPRC on this dataset.
- **Benford features give a small but visible lift** when added on top of LR. They are not strong enough to beat gradient-boosted trees, but they are a near-zero-cost addition and they carry interpretive value: a fraud-conditional leading-digit distribution that deviates visibly from Benford is itself a reportable finding.
- **Time-window aggregates catch the sequence-flavored patterns** Stripe highlights. On this dataset the effect is modest because we do not have per-card grouping (the classic ULB dataset has no card ID), so our windows aggregate globally rather than per user.
- **Gradient-boosted trees are the hard baseline to beat.** Both XGBoost and LightGBM consistently dominate LR and usually match or exceed the neural approaches on tabular fraud data of this size.
- **The autoencoder is a genuinely different family** — it never sees fraud during training and scores by reconstruction error. It is rarely the best model by AUPRC on this data, but it is a legitimate unsupervised comparison and it shines at recall at low precision.

### Limitations
- **PCA-anonymized features.** V1–V28 are already principal components, so domain-aware feature engineering is impossible. Any gains have to come from the modeling side.
- **No per-card grouping.** The ULB dataset does not carry a card or user identifier, so we cannot reproduce Stripe's *per-entity sequence* signal. Our time-window features aggregate globally, which is strictly weaker.
- **Single dataset.** Results here do not generalize to modern payments fraud, which differs in attack patterns, amount distributions, and the ratio of card-testing to other fraud types.
- **Time-ordered split is an 80/20 cut, not a rolling evaluation.** Real production systems retrain continuously; our test set is a single held-out window.
- **No hyperparameter sweep.** Every model uses fixed reasonable defaults. A real submission would include a per-model validation-set search.